# RAG - Penyakit Manusia    

In [ ]:
%pip install -U -q "google-genai>=2.9.0" 

Note: you may need to restart the kernel to use updated packages.


## Config


In [ ]:
from google import genai
from dotenv import load_dotenv
import os
from supabase import create_client, Client
from pathlib import Path


load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

clientGemini = genai.Client(api_key = gemini_api_key)

supabase: Client = create_client(
    os.environ.get("SUPABASE_URL"),
    os.environ.get("SUPABASE_KEY")
)

KNOWLEDGE_DIR = Path("knowledge")

## Chunking

In [6]:
def chunk_text(text: str, chunk_size=1000, overlap=200):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

In [7]:
def load_markdown_files():

    files = list(KNOWLEDGE_DIR.rglob("*.md"))

    for file_path in files:

        disease = file_path.parent.name
        category = file_path.stem

        content = file_path.read_text(
            encoding="utf-8"
        )

        yield {
            "file_path": file_path,
            "disease": disease,
            "category": category,
            "content": content
        }

In [ ]:
# from google.genai import types

def generate_embedding(text: str):

    response = clientGemini.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config={
            "output_dimensionality": 768
        }
    )

    return response.embeddings[0].values

In [18]:
def insert_document(doc):

    result = supabase.table("documents").insert({
        "title": f"{doc['disease']} - {doc['category']}",
        "source": "WHO",
        "category": doc["category"],
        "contents": doc["content"]
    }).execute()

    return result.data[0]["id"]

In [11]:
def insert_chunks(document_id, doc):

    chunks = chunk_text(doc["content"])

    rows = []

    for index, chunk in enumerate(chunks):

        embedding = generate_embedding(chunk)

        rows.append({
            "document_id": document_id,
            "content": chunk,
            "chunk_index": index,
            "embedding": embedding,
            "metadata": {
                "disease": doc["disease"],
                "category": doc["category"],
                "language": "id"
            }
        })

    supabase.table("document_chunks").insert(rows).execute()

In [16]:
def main():

    for doc in load_markdown_files():

        print(
            f"Processing: {doc['file_path']}"
        )

        document_id = insert_document(doc)

        insert_chunks(
            document_id,
            doc
        )

        print(
            f"✓ Completed: {doc['file_path']}"
        )



In [21]:

if __name__ == "__main__":
    main()

Processing: knowledge\covid\corona_virus.md
✓ Completed: knowledge\covid\corona_virus.md
Processing: knowledge\covid\covid_disease.md
✓ Completed: knowledge\covid\covid_disease.md
Processing: knowledge\pneumonia\children_pneumonia.md
✓ Completed: knowledge\pneumonia\children_pneumonia.md
Processing: knowledge\pneumonia\overview.md
✓ Completed: knowledge\pneumonia\overview.md
Processing: knowledge\tbc\fact.md
✓ Completed: knowledge\tbc\fact.md
Processing: knowledge\tbc\overview.md
✓ Completed: knowledge\tbc\overview.md
Processing: knowledge\tbc\pencegahan.md
✓ Completed: knowledge\tbc\pencegahan.md
Processing: knowledge\tbc\tbc.md
✓ Completed: knowledge\tbc\tbc.md
Processing: knowledge\tbc\wpro_tbc.md
✓ Completed: knowledge\tbc\wpro_tbc.md


## TESTING

In [27]:
def generate_answer(question, chunks):
    context = "\n\n".join(chunks)

    prompt = f"""
Kamu adalah asisten informasi kesehatan.

Jawab pertanyaan pengguna hanya berdasarkan informasi
yang diberikan pada CONTEXT.

Jika informasi tidak tersedia dalam CONTEXT, katakan
bahwa informasi tersebut tidak ditemukan dalam sumber.

Jangan membuat diagnosis atau memberikan kepastian
medis kepada pengguna.

CONTEXT:
{context}

PERTANYAAN:
{question}

JAWABAN:
"""

    response = clientGemini.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

In [28]:
query = "Apa gejala penyakit TBC?"

response = clientGemini.models.embed_content(
    model="gemini-embedding-001",
    contents=query,
    config={
        "output_dimensionality": 768
    }
)

query_embedding = response.embeddings[0].values

result = supabase.rpc(
    "match_document_chunks",
    {
        "query_embedding": query_embedding,
        "match_threshold": 0.3,
        "match_count": 5
    }
).execute()

chunks = [
    row["content"]
    for row in result.data
]

answer = generate_answer(
    query,
    chunks
)

print(answer)

Berdasarkan informasi yang diberikan dalam sumber, gejala umum penyakit TBC (tuberkulosis) meliputi:

- Batuk berkepanjangan atau persisten (terkadang disertai dahak atau darah)
- Nyeri dada
- Kelemahan
- Kelelahan
- Penurunan berat badan (yang tidak disengaja)
- Demam
- Keringat malam

Gejala yang dialami seseorang bergantung pada bagian tubuh yang terdampak (seperti paru-paru, ginjal, otak, tulang belakang, atau kulit). 

Selain itu, beberapa orang yang terinfeksi penyakit TBC mungkin tidak memiliki gejala sama sekali, atau gejalanya bersifat ringan pada awalnya dan berkembang secara perlahan.
